<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/2026-2027/2_1_Logistic_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part II. Classification. Logistic Regression

# Глава 4. Логистическая регрессия (часть I)

В предыдущих главах мы предсказывали непрерывную величину — цену квартиры или стоимость автомобиля. Но часто ответ — это «да» или «нет». Вернёт ли заёмщик кредит? Является ли письмо спамом? Уйдёт ли клиент к конкуренту? Для таких задач используется **логистическая регрессия**. Она переносит все идеи линейной регрессии — линейную комбинацию признаков, принцип максимального правдоподобия, градиентную оптимизацию — в мир вероятностей, и остаётся одним из самых интерпретируемых и надёжных инструментов бинарной классификации.

## 1. Постановка задачи бинарной классификации

Пусть для каждого объекта мы наблюдаем вектор признаков $\mathbf{x} \in \mathbb{R}^D$ и бинарный отклик
$$
y \in \{0, 1\}.
$$
Единица кодирует интересующее нас событие (например, дефолт заёмщика), ноль — его отсутствие. Мы хотим построить модель, которая по вектору $\mathbf{x}$ предсказывает не просто класс, а **вероятность** принадлежности к классу $1$:
$$
P(y = 1 \mid \mathbf{x}).
$$

Почему нельзя просто применить линейную регрессию $t = \mathbf{w}^T \mathbf{x} + \varepsilon$ к такой задаче?  
Во-первых, линейная функция неограничена и может выдавать значения далеко за пределами отрезка $[0,1]$, тогда как вероятность обязана лежать именно в этих границах.  
Во-вторых, модель линейной регрессии предполагает постоянную дисперсию ошибок (гомоскедастичность). Для бинарного отклика дисперсия $y$ при фиксированном $\mathbf{x}$ равна $p(1-p)$, где $p = P(y=1|\mathbf{x})$, и она меняется с изменением $p$, принимая максимальное значение при $p=0.5$ и стремясь к нулю на краях. Таким образом, гомоскедастичность заведомо нарушена, и стандартные ошибки МНК-оценок будут некорректны.

Выход заключается в том, чтобы моделировать не сам отклик, а вероятность $p$ через нелинейное преобразование линейной комбинации признаков. Ключевой элемент этого преобразования — **сигмоидная функция** (логистическая функция):
$$
\sigma(z) = \frac{1}{1 + e^{-z}}.
$$

Её график изображён на рисунке 1 (слева). Видно, что функция плавно переходит от 0 к 1, пересекая значение 0.5 в точке z=0. Справа показана производная сигмоиды, достигающая максимума в нуле — это область наибольшей неопределённости модели.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Создаём диапазон значений z
z = np.linspace(-7, 7, 200)
sigma = 1 / (1 + np.exp(-z))
d_sigma = sigma * (1 - sigma)

# Настройка графиков
plt.figure(figsize=(12, 4))

# График 1: сигмоида
plt.subplot(1, 2, 1)
plt.plot(z, sigma, 'b-', linewidth=2, label=r'$\sigma(z) = \frac{1}{1+e^{-z}}$')
plt.axhline(0, color='gray', linestyle='--', alpha=0.5)
plt.axhline(1, color='gray', linestyle='--', alpha=0.5)
plt.axvline(0, color='gray', linestyle='--', alpha=0.5)
plt.xlabel('z')
plt.ylabel('σ(z)')
plt.title('Сигмоидная функция')
plt.legend()
plt.grid(alpha=0.3)

# График 2: производная сигмоиды
plt.subplot(1, 2, 2)
plt.plot(z, d_sigma, 'r-', linewidth=2, label=r"$\sigma'(z) = \sigma(z)(1-\sigma(z))$")
plt.xlabel('z')
plt.ylabel("σ'(z)")
plt.title('Производная сигмоиды')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

Разберём свойства сигмоиды подробно.

- **Область значений.** $\sigma(z) \in (0,1)$ для любого $z \in \mathbb{R}$. При $z \to -\infty$ $\sigma(z) \to 0$, при $z \to +\infty$ $\sigma(z) \to 1$. Это идеально подходит для моделирования вероятности.
- **Монотонность.** Функция строго возрастает: чем больше линейная комбинация, тем выше вероятность класса 1.
- **Значение в нуле.** $\sigma(0) = 1/(1+1) = 0.5$. Точка симметрии.
- **Симметричность.** Нетрудно проверить, что $\sigma(-z) = 1 - \sigma(z)$. Вероятность дополнительного события равна единице минус вероятность исходного.

Наиболее важное свойство — **производная сигмоиды**, которая выражается через саму функцию:
$$
\sigma'(z) = \frac{d}{dz}\left(\frac{1}{1+e^{-z}}\right)
= \frac{e^{-z}}{(1+e^{-z})^2}
= \frac{1}{1+e^{-z}} \cdot \frac{e^{-z}}{1+e^{-z}}
= \sigma(z)(1 - \sigma(z)).
$$
Это компактное выражение чрезвычайно упростит вычисление градиентов в дальнейшем.

> **Углублённый взгляд: строгий вывод связи логита и сигмоиды.**  
> Пусть $p = \sigma(z) = \frac{1}{1+e^{-z}}$. Выразим $z$ через $p$:
> $$
> 1+e^{-z} = \frac{1}{p} \quad\Rightarrow\quad e^{-z} = \frac{1-p}{p} \quad\Rightarrow\quad e^{z} = \frac{p}{1-p}.
> $$
> Логарифмируя, получаем
> $$
> z = \ln\frac{p}{1-p} \equiv \text{logit}(p).
> $$
> Обратно, если $z = \ln\frac{p}{1-p}$, то $\frac{p}{1-p} = e^{z}$, откуда $p = \frac{e^{z}}{1+e^{z}} = \frac{1}{1+e^{-z}} = \sigma(z)$.  
> Таким образом, **сигмоида и логит — взаимно обратные функции**. Логит переводит вероятность $p\in(0,1)$ в вещественное число $z\in\mathbb{R}$, а сигмоида — наоборот.
>
> Свойства обеих функций удобно свести в таблицу:
>
> | Функция | Обозначение | Область определения | Область значений | Монотонность | Симметрия |
> |---------|--------------|---------------------|------------------|--------------|------------|
> | Сигмоида | $\sigma(z)$ | $\mathbb{R}$ | $(0,1)$ | Возрастает | $\sigma(-z)=1-\sigma(z)$ |
> | Логит | $\text{logit}(p)$ | $(0,1)$ | $\mathbb{R}$ | Возрастает | $\text{logit}(1-p) = -\text{logit}(p)$ |
>
> Эта двойственность лежит в основе интерпретации коэффициентов модели.



Итак, модель логистической регрессии принимает вид
$$
P(y = 1 \mid \mathbf{x}) = \sigma\!\bigl(\mathbf{w}^T \boldsymbol{\phi}(\mathbf{x})\bigr),
$$
где $\boldsymbol{\phi}(\mathbf{x})$ — вектор базисных функций (как и в линейной регрессии, он может включать в себя единицу для свободного члена, полиномиальные и категориальные признаки). Величину $\mathbf{w}^T \boldsymbol{\phi}(\mathbf{x})$ часто называют **логитом** (logit) или логарифмом шансов (log-odds). В матричной форме для всей выборки
$$
\mathbf{p} = \sigma(\Phi \mathbf{w}),
$$
где $\Phi$ — матрица плана размера $n \times M$, а сигмоида применяется поэлементно.

## 2. Отношение шансов и интерпретация коэффициентов

Прелесть логистической регрессии — в её исключительной интерпретируемости. Введём **шансы** (odds) события:
$$
\text{odds} = \frac{P(y=1)}{P(y=0)} = \frac{p}{1-p}.
$$
Шансы показывают, во сколько раз вероятность успеха больше вероятности неудачи. Например, если $p = 0.75$, шансы равны $0.75/0.25 = 3$, т.е. успех в три раза вероятнее неудачи.

Из предыдущего раздела мы знаем, что
$$
\ln(\text{odds}) = \text{logit}(p) = \mathbf{w}^T \boldsymbol{\phi}(\mathbf{x}).
$$
Это ключевое соотношение: **логарифм шансов линеен по параметрам**. Следовательно, сами шансы мультипликативно зависят от признаков:
$$
\frac{p}{1-p} = \exp\!\bigl(\mathbf{w}^T \boldsymbol{\phi}(\mathbf{x})\bigr) = e^{w_0} \cdot e^{w_1\phi_1} \cdots e^{w_M\phi_M}.
$$

Теперь интерпретация коэффициента $w_j$ становится прозрачной. Зафиксируем все признаки, кроме $j$-го, и увеличим $\phi_j$ на единицу. Тогда шансы умножаются на $e^{w_j}$:
$$
\frac{p_{\text{нов}}}{1-p_{\text{нов}}} = \frac{p_{\text{стар}}}{1-p_{\text{стар}}} \cdot e^{w_j}.
$$
- Если $w_j > 0$, то $e^{w_j} > 1$ — шансы возрастают, признак повышает вероятность события.
- Если $w_j < 0$, то $e^{w_j} < 1$ — шансы уменьшаются, признак действует как протективный фактор.
- При $w_j = 0$ имеем $e^0 = 1$, шансы не меняются.

Для малых по модулю коэффициентов ($|w_j| \ll 1$) справедливо приближение $e^{w_j} \approx 1 + w_j$, то есть шансы изменяются примерно на $w_j \times 100\%$. Эта наглядность делает логистическую регрессию незаменимой в медицине («фактор курения увеличивает шансы заболевания в 1.8 раза»), в кредитном скоринге («каждый дополнительный год кредитной истории снижает шансы дефолта на 12%») и в социальных науках, где важно не только предсказать, но и объяснить.

## 3. Метод максимального правдоподобия

Как и в линейной регрессии, для настройки весов $\mathbf{w}$ мы обращаемся к принципу максимального правдоподобия. Каждое наблюдение $y_i$ при заданном $\mathbf{x}_i$ представляет собой испытание Бернулли с вероятностью успеха $p_i = \sigma(\mathbf{w}^T \boldsymbol{\phi}(\mathbf{x}_i))$:
$$
y_i \sim \text{Bernoulli}(p_i).
$$
Вероятность получить конкретное значение $y_i$ равна $p_i^{y_i} (1-p_i)^{1-y_i}$. Поскольку наблюдения независимы, функция правдоподобия всей выборки есть произведение:
$$
L(\mathbf{w}) = \prod_{i=1}^{n} p_i^{y_i} (1-p_i)^{1-y_i}.
$$
Переходим к логарифмической функции правдоподобия — она монотонна, поэтому максимум не смещается:
$$
\ell(\mathbf{w}) = \sum_{i=1}^{n} \bigl[ y_i \ln p_i + (1-y_i) \ln (1-p_i) \bigr].
$$
В машинном обучении принято минимизировать функцию потерь, поэтому определим **кросс-энтропийную функцию потерь** (log loss) как отрицательное логарифмическое правдоподобие:
$$
E(\mathbf{w}) = -\ell(\mathbf{w}) = -\sum_{i=1}^{n} \bigl[ y_i \ln p_i + (1-y_i) \ln(1-p_i) \bigr], \quad p_i = \sigma(\mathbf{w}^T \boldsymbol{\phi}_i). \tag{3.1}
$$
Если модель предсказывает $p_i$, близкую к истинной метке $y_i$, то потери малы; если же она уверенно ошибается (предсказывает $p_i \approx 0$ при $y_i = 1$), одно из слагаемых стремится к $-\ln 0 = +\infty$, жёстко штрафуя модель.

К сожалению, в отличие от линейной регрессии, приравнивание градиента $E(\mathbf{w})$ к нулю не даёт замкнутого аналитического решения — из-за нелинейности сигмоиды. Мы вынуждены применять итеративные методы оптимизации, такие как градиентный спуск.

> **Углублённый взгляд: строгое доказательство выпуклости кросс-энтропии.**  
> Вычислим гессиан $E(\mathbf{w})$. Градиент по $\mathbf{w}$ уже известен (см. раздел 4): $\nabla E = \Phi^T (\mathbf{p} - \mathbf{y})$. Продифференцируем его по $\mathbf{w}^T$:
> $$
> H = \frac{\partial}{\partial \mathbf{w}} \Phi^T (\mathbf{p} - \mathbf{y}) = \Phi^T \frac{\partial \mathbf{p}}{\partial \mathbf{w}}.
> $$
> Так как $\mathbf{p} = \sigma(\Phi \mathbf{w})$ и $\sigma'(z) = \sigma(z)(1-\sigma(z))$, то
> $$
> \frac{\partial p_i}{\partial w_j} = p_i(1-p_i) \Phi_{ij},
> $$
> что в матричном виде даёт $\frac{\partial \mathbf{p}}{\partial \mathbf{w}} = D \Phi$, где $D = \text{diag}\bigl(p_i(1-p_i)\bigr)$.
> Следовательно,
> $$
> H = \Phi^T D \Phi.
> $$
> Поскольку $0 < p_i < 1$, все $p_i(1-p_i) > 0$, и матрица $D$ положительно определена. Тогда для любого ненулевого вектора $\mathbf{v}$ имеем $\mathbf{v}^T H \mathbf{v} = (\Phi \mathbf{v})^T D (\Phi \mathbf{v}) \ge 0$, причём равенство достигается лишь если $\Phi \mathbf{v} = \mathbf{0}$. При полном столбцовом ранге $\Phi$ это означает $\mathbf{v} = \mathbf{0}$, и $H$ строго положительно определена. Таким образом, **кросс-энтропия является строго выпуклой функцией $\mathbf{w}$**, и любой найденный минимум будет глобальным. Это гарантирует устойчивость оптимизации.

> **Углублённый взгляд: численный пример вычисления кросс-энтропии.**  
> Чтобы дать осязаемое понимание функции потерь, вычислим её вручную для крошечного набора данных. Пусть имеется 6 объектов с тремя бинарными признаками $x_1, x_2, x_3 \in \{0,1\}$ и известными метками $y$. Начальные веса заданы как $\mathbf{w} = (0.1,\, -0.1,\, 0.2)$, свободный член $b = 0$.  
>
> | $i$ | $x_1$ | $x_2$ | $x_3$ | $y$ |
> |-----|-------|-------|-------|-----|
> | 1   | 1     | 0     | 1     | 0   |
> | 2   | 0     | 1     | 0     | 1   |
> | 3   | 1     | 0     | 0     | 0   |
> | 4   | 1     | 1     | 0     | 1   |
> | 5   | 0     | 0     | 0     | 0   |
> | 6   | 0     | 1     | 1     | 1   |
>
> Для каждого наблюдения вычислим линейную комбинацию $z^{(i)} = 0.1 x_1 - 0.1 x_2 + 0.2 x_3$, предсказанную вероятность $\hat{y}^{(i)} = \sigma(z^{(i)})$ и индивидуальные потери $-\bigl[y^{(i)}\ln\hat{y}^{(i)} + (1-y^{(i)})\ln(1-\hat{y}^{(i)})\bigr]$.
>
> - Объект 1: $z = 0.1\cdot1 - 0.1\cdot0 + 0.2\cdot1 = 0.3$, $\sigma(0.3) \approx 0.5744$, $y=0$, потеря $= -\ln(1-0.5744) = -\ln 0.4256 \approx 0.8544$.
> - Объект 2: $z = 0 - 0.1\cdot1 + 0 = -0.1$, $\sigma(-0.1) \approx 0.4750$, $y=1$, потеря $= -\ln 0.4750 \approx 0.7444$.
> - Объект 3: $z = 0.1$, $\sigma(0.1) \approx 0.5250$, $y=0$, потеря $= -\ln(1-0.5250) = -\ln 0.4750 \approx 0.7444$.
> - Объект 4: $z = 0.1 - 0.1 + 0 = 0$, $\sigma(0) = 0.5$, $y=1$, потеря $= -\ln 0.5 \approx 0.6931$.
> - Объект 5: $z = 0$, $\sigma(0) = 0.5$, $y=0$, потеря $= -\ln 0.5 \approx 0.6931$.
> - Объект 6: $z = 0 - 0.1\cdot1 + 0.2\cdot1 = 0.1$, $\sigma(0.1) \approx 0.5250$, $y=1$, потеря $= -\ln 0.5250 \approx 0.6444$.
>
> Суммируя шесть значений, получаем общую кросс-энтропию
> $$
> J = 0.8544 + 0.7444 + 0.7444 + 0.6931 + 0.6931 + 0.6444 = 4.3738 \approx 4.374.
> $$
> Этот пример наглядно показывает, как «уверенная» ошибка (объект 1, где модель выдала $p \approx 0.57$ при истинном $y=0$) вносит наибольший вклад в потери, в то время как «колеблющиеся» предсказания в районе $0.5$ дают потери порядка $0.69$.


На рисунке ниже (слева) показано, как кросс-энтропия штрафует модель в зависимости от предсказанной вероятности. Для истинного класса $y=1$ потеря стремится к бесконечности при $p \to 0$ (модель уверена в противоположном классе). Для $y=0$ потеря растёт при $p \to 1$. На правом графике виден вклад каждого из шести объектов примера: объект 1 (y=0, p≈0.57) даёт наибольшую ошибку, так как модель «уверенно» ошибается.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Данные из численного примера
y_true = np.array([0, 1, 0, 1, 0, 1])
p_pred = np.array([0.5744, 0.4750, 0.5250, 0.5000, 0.5000, 0.5250])
losses = np.array([0.8544, 0.7444, 0.7444, 0.6931, 0.6931, 0.6444])

# График 1: функция потерь для одного наблюдения
p_range = np.linspace(0.001, 0.999, 200)
loss_when_y1 = -np.log(p_range)
loss_when_y0 = -np.log(1 - p_range)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(p_range, loss_when_y1, 'r-', linewidth=2, label='y = 1 (потеря = -ln p)')
plt.plot(p_range, loss_when_y0, 'b-', linewidth=2, label='y = 0 (потеря = -ln(1-p))')
plt.axvline(0.5, color='gray', linestyle='--', alpha=0.5, label='p = 0.5')
plt.xlabel('Предсказанная вероятность p')
plt.ylabel('Индивидуальная потеря')
plt.title('Кросс-энтропия для одного наблюдения')
plt.legend()
plt.grid(alpha=0.3)

# График 2: вклад каждого объекта из примера
plt.subplot(1, 2, 2)
colors = ['red' if y == 0 else 'blue' for y in y_true]
bars = plt.bar(range(1, 7), losses, color=colors, alpha=0.7, edgecolor='black')
plt.xticks(range(1, 7))
plt.xlabel('Номер объекта')
plt.ylabel('Потеря')
plt.title(f'Вклад каждого объекта в кросс-энтропию\nОбщая потеря = {sum(losses):.3f}')
plt.legend([bars[0], bars[1]], ['y = 0 (ошибка дороже?)', 'y = 1'], loc='upper right')
plt.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()



## 4. Градиент функции потерь

Для оптимизации $E(\mathbf{w})$ градиентным спуском нам потребуется градиент. Вычислим частную производную по одному весу $w_k$:
$$
\frac{\partial E}{\partial w_k}
= -\sum_{i=1}^{n} \left[ y_i \frac{1}{p_i}\frac{\partial p_i}{\partial w_k} + (1-y_i) \frac{1}{1-p_i}\left(-\frac{\partial p_i}{\partial w_k}\right) \right].
$$
Производная сигмоиды даёт $\frac{\partial p_i}{\partial w_k} = p_i(1-p_i) \phi_k(\mathbf{x}_i)$. Подставляя,
$$
\frac{\partial E}{\partial w_k}
= -\sum_{i=1}^{n} \left[ y_i (1-p_i) - (1-y_i) p_i \right] \phi_k(\mathbf{x}_i).
$$
Выражение в скобках упрощается до $y_i - p_i$. Действительно, $y_i(1-p_i) - (1-y_i)p_i = y_i - y_i p_i - p_i + y_i p_i = y_i - p_i$. Таким образом,
$$
\boxed{\frac{\partial E}{\partial w_k} = \sum_{i=1}^{n} (p_i - y_i)\,\phi_k(\mathbf{x}_i)}. \tag{4.1}
$$
Соберём все частные производные в вектор:
$$
\nabla_{\mathbf{w}} E = \sum_{i=1}^{n} (p_i - y_i)\,\boldsymbol{\phi}(\mathbf{x}_i).
$$
В матричной форме это принимает элегантный вид:
$$
\boxed{\nabla_{\mathbf{w}} E = \Phi^T (\mathbf{p} - \mathbf{y})}. \tag{4.2}
$$
Сравним с градиентом линейной регрессии $\Phi^T(\Phi\mathbf{w} - \mathbf{t})$. Форма очень похожа: везде фигурирует разность предсказаний и истинных значений. Но есть фундаментальное отличие: в линейной регрессии предсказания $\Phi\mathbf{w}$ линейны по $\mathbf{w}$, и градиент линеен, тогда как здесь $\mathbf{p} = \sigma(\Phi\mathbf{w})$ нелинейно зависит от $\mathbf{w}$. Именно поэтому не существует явной формулы для $\hat{\mathbf{w}}$, и мы вынуждены применять итеративные алгоритмы.

Градиент (4.2) устроен чрезвычайно удобно для вычислений: матричное умножение $\Phi^T(\mathbf{p} - \mathbf{y})$ требует лишь одного прохода по данным и легко распараллеливается.

> **Углублённый взгляд: проблема полной разделимости.**  
> Если обучающая выборка линейно разделима (существует гиперплоскость, идеально отделяющая классы), то можно добиться $p_i \to 1$ для $y_i=1$ и $p_i \to 0$ для $y_i=0$, неограниченно увеличивая норму вектора $\mathbf{w}$. При этом $E(\mathbf{w}) \to 0$, но оптимальное решение в смысле максимума правдоподобия отсутствует — веса стремятся к бесконечности. На практике это приводит к численной нестабильности и гигантским коэффициентам. Решением служит **регуляризация** ($L_1$ или $L_2$), которая стабилизирует оптимизацию и предотвращает переобучение. Мы вернёмся к этому во второй части главы.

---

## Вопросы для самопроверки

### Для начинающих

1. Почему для бинарного отклика $y \in \{0,1\}$ нельзя напрямую применить обычную линейную регрессию? Назовите не менее двух причин.
2. Запишите уравнение сигмоидной функции и перечислите три её ключевых свойства (область значений, симметричность, вид производной).
3. Что такое логит и как он связан с шансами события? Если шансы события равны 4, чему равна его вероятность?
4. В чём состоит метод максимального правдоподобия применительно к логистической регрессии? Какая функция потерь при этом получается?
5. Выпишите выражение для градиента кросс-энтропийной функции потерь в матричной форме. Чем оно похоже на градиент линейной регрессии и чем отличается?
6. Интерпретируйте коэффициент $w_1 = -0.5$ в логистической регрессии: что произойдёт с шансами при увеличении соответствующего признака на единицу?

### Для профессионалов

1. Докажите, что сигмоида $\sigma(z) = 1/(1+e^{-z})$ и функция логита $\text{logit}(p) = \ln\frac{p}{1-p}$ являются взаимно обратными. Какая из них переводит $\mathbb{R}$ в $(0,1)$, а какая — наоборот?
2. Выведите гессиан функции кросс-энтропии и покажите, что он положительно полуопределён. При каком условии на матрицу плана $\Phi$ гессиан становится строго положительно определённым, а функция потерь — строго выпуклой?
3. В чём заключается проблема полной разделимости классов в логистической регрессии? Почему в этом случае оценки максимального правдоподобия не существуют как конечный вектор?
4. Используя выражение для градиента $\nabla E = \Phi^T(\mathbf{p} - \mathbf{y})$, предложите правило обновления весов для пакетного градиентного спуска. Как изменится это правило при переходе к стохастическому градиентному спуску (SGD)?
5. Свяжите логистическую регрессию с обобщёнными линейными моделями (GLM): какую функцию связи и какое распределение отклика она использует? Почему каноническая функция связи для биномиального распределения — именно логит?


# 5. Численные методы оптимизации

В отличие от линейной регрессии, где мы получили явную формулу $\hat{\mathbf{w}} = (\Phi^T\Phi)^{-1}\Phi^T\mathbf{t}$, логистическая регрессия не имеет аналитического решения. Её функция потерь — кросс-энтропия — строго выпукла, но нелинейна по параметрам, и минимум приходится искать итеративно. К счастью, градиент имеет простой вид $\nabla E = \Phi^T (\boldsymbol{\sigma} - \mathbf{y})$, что позволяет эффективно применять весь арсенал градиентных методов, освоенных нами в предыдущей главе.




## 5.1 Пакетный градиентный спуск

Самая прямая стратегия — на каждом шаге вычислять градиент по всей обучающей выборке и сдвигать веса в противоположном направлении:

$$
\mathbf{w}^{(k+1)} = \mathbf{w}^{(k)} - \alpha \Phi^T \bigl(\boldsymbol{\sigma}^{(k)} - \mathbf{y}\bigr), \tag{5.1}
$$

где $\boldsymbol{\sigma}^{(k)} = \sigma(\Phi \mathbf{w}^{(k)})$ — вектор предсказанных вероятностей на текущей итерации, а $\alpha > 0$ — скорость обучения. Этот метод называют **пакетным градиентным спуском** (batch gradient descent). Он гарантированно сходится к глобальному минимуму благодаря выпуклости кросс-энтропии, но требует на каждом шаге вычислений по всей выборке, что при больших $n$ может быть медленным.

Выбор $\alpha$ критичен: слишком маленький шаг ведёт к медленной сходимости, слишком большой — к осцилляциям или расходимости. На практике $\alpha$ подбирают эмпирически, отслеживая динамику функции потерь на обучающей и валидационной выборках.

На рисунке показана траектория пакетного градиентного спуска на фоне линий уровня кросс-энтропии для двух весов (свободного члена $w_0$ и коэффициента при первом признаке $w_1$). В отличие от квадратичного ландшафта MSE в линейной регрессии (эллипсоиды), линии уровня кросс-энтропии имеют более сложную форму, но остаются выпуклыми. Алгоритм движется к минимуму, постепенно замедляясь по мере приближения к нему — градиент уменьшается. При правильно выбранной скорости обучения $\alpha = 0.5$ траектория плавная и не осциллирует.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# --- Генерация простых двумерных данных для визуализации ---
np.random.seed(0)
n_vis = 100
X_vis = np.random.randn(n_vis, 2)
# Истинная разделяющая граница: w1*x1 + w2*x2 + b = 0
true_w = np.array([2.0, -1.5])
true_b = 0.5
z_true = X_vis @ true_w + true_b
p_true = 1 / (1 + np.exp(-z_true))
y_vis = np.random.binomial(1, p_true)

# Добавляем столбец единиц для свободного члена
X_vis_aug = np.hstack([np.ones((n_vis, 1)), X_vis])

# --- Функция кросс-энтропии и её градиент ---
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def cross_entropy(w, X, y):
    p = sigmoid(X @ w)
    p = np.clip(p, 1e-15, 1 - 1e-15)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def grad_cross_entropy(w, X, y):
    p = sigmoid(X @ w)
    return X.T @ (p - y) / len(y)

# --- Вычисление ландшафта кросс-энтропии ---
w0_range = np.linspace(-3, 5, 60)
w1_range = np.linspace(-4, 4, 60)
W0, W1 = np.meshgrid(w0_range, w1_range)
Z_loss = np.zeros_like(W0)
for i in range(len(w0_range)):
    for j in range(len(w1_range)):
        w_test = np.array([W0[i, j], W1[i, j], 0.0])
        Z_loss[j, i] = cross_entropy(w_test, X_vis_aug, y_vis)

# --- Градиентный спуск с сохранением траектории ---
w = np.array([0.0, 0.0, 0.0])  # w0, w1, w2
alpha = 0.5
n_steps = 30
trajectory = [(w[0], w[1])]

for _ in range(n_steps):
    grad = grad_cross_entropy(w, X_vis_aug, y_vis)
    w -= alpha * grad
    trajectory.append((w[0], w[1]))
trajectory = np.array(trajectory)

# --- Рисунок 1: 3D-ландшафт и линии уровня ---
fig = plt.figure(figsize=(14, 5))

# 3D поверхность
ax1 = fig.add_subplot(121, projection='3d')
surf = ax1.plot_surface(W0, W1, Z_loss, cmap='viridis', alpha=0.8, linewidth=0, antialiased=True)
ax1.set_xlabel('w₀ (свободный член)', fontsize=10)
ax1.set_ylabel('w₁ (коэффициент)', fontsize=10)
ax1.set_zlabel('Кросс-энтропия', fontsize=10)
ax1.set_title('3D-ландшафт кросс-энтропии')
fig.colorbar(surf, ax=ax1, shrink=0.5, aspect=10)

# Контурный график
ax2 = fig.add_subplot(122)
contour = ax2.contour(W0, W1, Z_loss, levels=40, cmap='viridis', alpha=0.7)
ax2.set_xlabel('w₀')
ax2.set_ylabel('w₁')
ax2.set_title('Линии уровня кросс-энтропии')
fig.colorbar(contour, ax=ax2, shrink=0.5)

plt.suptitle('Рисунок X: Ландшафт функции потерь логистической регрессии', fontsize=12)
plt.tight_layout()
plt.show()

# --- Рисунок 2: Траектория градиентного спуска на линиях уровня ---
plt.figure(figsize=(10, 8))
contour = plt.contour(W0, W1, Z_loss, levels=50, cmap='viridis', alpha=0.7)
plt.colorbar(contour, label='Кросс-энтропия')
plt.plot(trajectory[:, 0], trajectory[:, 1], 'r.-', linewidth=1.5, markersize=6, label='Траектория GD')
plt.plot(trajectory[0, 0], trajectory[0, 1], 'bo', markersize=10, label='Старт')
plt.plot(trajectory[-1, 0], trajectory[-1, 1], 'r*', markersize=12, label='Финиш')
plt.xlabel('w₀ (свободный член)')
plt.ylabel('w₁ (коэффициент при первом признаке)')
plt.title('Рисунок Y: Пакетный градиентный спуск на линиях уровня кросс-энтропии')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 5.2 Стохастический и mini-batch варианты

Для масштабных данных пакетный градиентный спуск становится неэффективным. Тогда переходят к **стохастическому градиентному спуску** (SGD): на каждой итерации выбирается одно случайное наблюдение $i$, и веса обновляются по его вкладу:

$$
\mathbf{w} \leftarrow \mathbf{w} - \alpha (p_i - y_i)\,\boldsymbol{\phi}(\mathbf{x}_i).
$$

Такой градиент — несмещённая, но очень зашумлённая оценка истинного градиента. Шум помогает избегать мелких локальных минимумов в невыпуклых задачах, но в строго выпуклой кросс-энтропии он в основном замедляет сходимость. Компромисс — **mini-batch GD**, где градиент усредняется по небольшой случайной подвыборке (батчу) размера $B$. Это снижает дисперсию оценки и позволяет использовать векторизованные вычисления, характерные для современных библиотек.

Для любого варианта SGD полезно постепенно уменьшать $\alpha$ по мере обучения. Распространённые расписания:
- **Step decay**: $\alpha$ умножается на коэффициент $\gamma$ каждые $T$ эпох.
- **Cosine annealing**: $\alpha_k = \alpha_{\min} + \frac{1}{2}(\alpha_{\max} - \alpha_{\min})(1 + \cos(\pi k / K))$, где $K$ — общее число итераций. Этот метод плавно уменьшает шаг, давая алгоритму возможность «осмотреться» вблизи минимума.

> **Углублённый взгляд: ручной расчёт одной итерации градиентного спуска**
>
> Продолжим наш игрушечный пример из первой части. Напомним: 6 объектов, начальные веса $\mathbf{w} = (0.1, -0.1, 0.2)$, свободный член $b = 0$, кросс-энтропия $J \approx 4.374$. Вычислим градиенты всех параметров **в явном виде** и сделаем один шаг с $\alpha = 0.1$.
>
> Выпишем ещё раз предсказанные вероятности и разности $p_i - y_i$:
>
> | $i$ | $p_i$ | $p_i - y_i$ |
> |------|---------|----------------|
> | 1 | 0.5744 |  0.5744 |
> | 2 | 0.4750 | -0.5250 |
> | 3 | 0.5250 |  0.5250 |
> | 4 | 0.5000 | -0.5000 |
> | 5 | 0.5000 |  0.5000 |
> | 6 | 0.5250 | -0.4750 |
>
> Градиенты вычисляются по формуле $\frac{\partial J}{\partial w_j} = \sum_i (p_i - y_i) x_{ij}$ (знак соответствует кросс-энтропии с отрицательным логарифмом правдоподобия, как в (3.1)). Выполнив суммирование для каждого признака и свободного члена, получаем:
>
> $$
> \frac{\partial J}{\partial w_1} = -0.3758,\quad
> \frac{\partial J}{\partial w_2} = -1.4752,\quad
> \frac{\partial J}{\partial w_3} = 0.6740,\quad
> \frac{\partial J}{\partial b} = 0.1490.
> $$
>
> *(Эти значения полностью соответствуют конкретной реализации примера и могут быть проверены прямым вычислением.)*
>
> Шаг градиентного спуска с $\alpha = 0.1$:
> $$
> w_1^{\text{new}} = 0.1 - 0.1 \cdot (-0.3758) = 0.13758 \approx 0.1376,
> $$
> $$
> w_2^{\text{new}} = -0.1 - 0.1 \cdot (-1.4752) = 0.04752 \approx 0.0475,
> $$
> $$
> w_3^{\text{new}} = 0.2 - 0.1 \cdot 0.6740 = 0.1326,
> $$
> $$
> b^{\text{new}} = 0 - 0.1 \cdot 0.1490 = -0.0149.
> $$
>
> Мы видим, что градиенты для $w_1$ и $w_2$ отрицательны, значит, эти веса должны быть **увеличены**, чтобы компенсировать первоначальную недооценку соответствующих признаков. Градиент для $w_3$ положителен — вес уменьшается. Свободный член сдвигается чуть вниз. После одного такого шага функция потерь уменьшится, и модель станет чуть лучше предсказывать обучающие метки.

> **Углублённый взгляд: метод Ньютона–Рафсона / IRLS**
>
> Метод второго порядка использует информацию о кривизне функции потерь, что позволяет делать более «умные» шаги. Для логистической регрессии гессиан равен $H = \Phi^T D \Phi$, где $D = \text{diag}(p_i(1-p_i))$ — положительно определённая матрица. Шаг Ньютона–Рафсона:
> $$
> \mathbf{w}^{(k+1)} = \mathbf{w}^{(k)} - H^{-1} \nabla E(\mathbf{w}^{(k)}).
> $$
> Подставляя выражения для градиента и гессиана, можно переписать обновление как решение задачи **взвешенного метода наименьших квадратов** (Iteratively Reweighted Least Squares, IRLS):
> $$
> \mathbf{w}^{(k+1)} = (\Phi^T D \Phi)^{-1} \Phi^T D \mathbf{z},
> $$
> где «рабочий отклик» $\mathbf{z} = \Phi \mathbf{w}^{(k)} - D^{-1}(\boldsymbol{\sigma} - \mathbf{y})$.
>
> Метод Ньютона обладает квадратичной сходимостью вблизи оптимума — число верных знаков удваивается на каждой итерации. Однако стоимость одной итерации — $O(M^3 + M^2 n)$ из-за обращения или решения системы с матрицей $M\times M$. Поэтому для очень больших $M$ предпочитают градиентные методы первого порядка, а для умеренных размерностей IRLS — превосходный выбор. Библиотеки типа `sklearn` по умолчанию используют квазиньютоновский L-BFGS, который аппроксимирует гессиан, балансируя скорость и масштабируемость.

# 6. Реализация с нуля на Python

Закрепим понимание, написав собственный класс логистической регрессии с пакетным градиентным спуском. В качестве данных возьмём Breast Cancer Wisconsin dataset: 569 образцов, 30 числовых признаков, цель — злокачественность опухоли (0 — malignant, 1 — benign).




In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score, roc_curve, roc_auc_score,
                             log_loss)
from scipy.special import expit as sigmoid  # устойчивая реализация сигмоиды

# Загрузка и подготовка данных
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Добавляем столбец единиц (свободный член)
X_train_aug = np.hstack([np.ones((X_train.shape[0], 1)), X_train_scaled])
X_test_aug  = np.hstack([np.ones((X_test.shape[0], 1)), X_test_scaled])

#Реализуем класс с методами `sigmoid`, `compute_loss`, `fit`, `predict_proba` и `predict`:

class LogisticRegressionGD:
    def __init__(self, lr=0.1, n_iter=1000, tol=1e-5):
        self.lr = lr
        self.n_iter = n_iter
        self.tol = tol
        self.w = None
        self.loss_history = []

    def sigmoid(self, z):
        return sigmoid(z)  # scipy.special.expit, устойчива к переполнению

    def compute_loss(self, X, y):
        p = self.sigmoid(X @ self.w)
        eps = 1e-15
        p = np.clip(p, eps, 1 - eps)
        return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

    def fit(self, X, y):
        n, m = X.shape
        self.w = np.zeros(m)
        for i in range(self.n_iter):
            p = self.sigmoid(X @ self.w)
            grad = X.T @ (p - y) / n
            self.w -= self.lr * grad
            loss = self.compute_loss(X, y)
            self.loss_history.append(loss)
            if i > 0 and abs(self.loss_history[-2] - loss) < self.tol:
                print(f"Сошлись на итерации {i}")
                break

    def predict_proba(self, X):
        return self.sigmoid(X @ self.w)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

#Обучим модель и нарисуем кривую сходимости:

model = LogisticRegressionGD(lr=0.5, n_iter=5000)
model.fit(X_train_aug, y_train)

plt.figure(figsize=(8,5))
plt.plot(model.loss_history)
plt.xlabel('Итерация')
plt.ylabel('Log Loss')
plt.title('Сходимость логистической регрессии (пакетный GD)')
plt.grid(alpha=0.3)
plt.show()


Теперь визуализируем вероятностную раскраску и разделяющую прямую. Чтобы можно было изобразить в двух измерениях, спроецируем данные на первые две главные компоненты:

In [ ]:
# Визуализация на двумерной проекции
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_train_scaled)

X_pca_aug = np.hstack([np.ones((X_pca.shape[0], 1)), X_pca])
model2d = LogisticRegressionGD(lr=0.5, n_iter=5000)
model2d.fit(X_pca_aug, y_train)

# Сетка для контуров
x1_min, x1_max = X_pca[:,0].min()-1, X_pca[:,0].max()+1
x2_min, x2_max = X_pca[:,1].min()-1, X_pca[:,1].max()+1
xx1, xx2 = np.meshgrid(np.linspace(x1_min, x1_max, 200),
                       np.linspace(x2_min, x2_max, 200))
grid = np.c_[np.ones(xx1.ravel().shape), xx1.ravel(), xx2.ravel()]
probs = model2d.predict_proba(grid).reshape(xx1.shape)

plt.figure(figsize=(8,6))
plt.contourf(xx1, xx2, probs, levels=20, cmap='RdBu', alpha=0.6)
plt.colorbar(label='P(y=1)')
plt.scatter(X_pca[y_train==0, 0], X_pca[y_train==0, 1], c='blue', label='Malignant')
plt.scatter(X_pca[y_train==1, 0], X_pca[y_train==1, 1], c='red', label='Benign')
# Разделяющая линия p=0.5
plt.contour(xx1, xx2, probs, levels=[0.5], colors='k', linewidths=2)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Логистическая регрессия: вероятностная раскраска')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

Сравним качество с эталонной реализацией `sklearn` без регуляризации (чтобы условия совпадали):

In [ ]:
sk_model = LogisticRegression(penalty=None, solver='lbfgs', max_iter=10000)
sk_model.fit(X_train_scaled, y_train)

print("Наша реализация:")
print(f"  Train accuracy: {accuracy_score(y_train, model.predict(X_train_aug)):.4f}")
print(f"  Test accuracy : {accuracy_score(y_test, model.predict(X_test_aug)):.4f}")
print("sklearn:")
print(f"  Train accuracy: {accuracy_score(y_train, sk_model.predict(X_train_scaled)):.4f}")
print(f"  Test accuracy : {accuracy_score(y_test, sk_model.predict(X_test_scaled)):.4f}")



Результаты будут близки, подтверждая корректность нашей реализации.

# 7. Метрики качества бинарной классификации

Одна лишь доля правильных ответов (accuracy) может вводить в заблуждение, особенно при несбалансированных классах. Если среди 1000 заёмщиков лишь 10 допускают дефолт, модель, всегда предсказывающая «нет дефолта», будет иметь accuracy 99%, но окажется бесполезной. Поэтому в бинарной классификации используют целую батарею метрик, основанных на **матрице ошибок** (confusion matrix).

Введём порог (обычно 0.5), превращающий вероятности в классы. Тогда объекты делятся на четыре категории:

- **TP** (True Positive) — верно предсказанный положительный класс (больной, спам);
- **TN** (True Negative) — верно предсказанный отрицательный;
- **FP** (False Positive) — ошибочно предсказанный положительный (ложная тревога);
- **FN** (False Negative) — пропущенный положительный (пропуск цели).

На основе этих четырёх чисел определяются основные метрики.

**Accuracy (доля правильных ответов)** — самая простая, но часто обманчивая мера:

$$
\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}.
$$

Она хорошо работает при сбалансированных классах, но при сильном дисбалансе может создавать иллюзию качества.

**Precision (точность)** — доля истинно положительных среди всех предсказанных положительных:

$$
\text{Precision} = \frac{TP}{TP + FP}.
$$

Эта метрика важна, когда цена ложной тревоги высока. Например, если модель рекомендует дорогостоящее лечение, мы хотим быть уверены, что пациент действительно болен.

**Recall (полнота, чувствительность)** — доля истинно положительных, которые модель смогла обнаружить:

$$
\text{Recall} = \frac{TP}{TP + FN}.
$$

Критична в задачах, где пропуск цели недопустим: скрининг рака, обнаружение мошеннических транзакций, поиск опасных объектов.

**F1-мера** — гармоническое среднее precision и recall, позволяющее сбалансировать эти две метрики:

$$
F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}.
$$

F1 принимает высокое значение только когда и точность, и полнота высоки одновременно. Она особенно полезна при сравнении моделей на несбалансированных данных.

Все эти метрики зависят от выбранного порога. Чтобы получить более полную картину, используют **ROC-кривую** (Receiver Operating Characteristic). Она строится путём варьирования порога и откладывания по оси $X$ доли ложных срабатываний, а по оси $Y$ — доли истинных срабатываний:

$$
\text{FPR} = \frac{FP}{FP + TN}, \qquad \text{TPR} = \frac{TP}{TP + FN} = \text{Recall}.
$$

Площадь под ROC-кривой называется **AUC** (Area Under the Curve). Она принимает значения от 0.5 (случайное гадание) до 1.0 (идеальное разделение). Существует элегантная вероятностная интерпретация:

$$
\text{AUC} = P(p_i > p_j \mid y_i = 1, y_j = 0),
$$

где $p_i, p_j$ — предсказанные вероятности для положительного и отрицательного объектов соответственно. Иными словами, AUC равна вероятности того, что случайно выбранный положительный объект получит более высокую оценку, чем случайно выбранный отрицательный.

Наконец, **log-loss** (логарифмическая потеря) напрямую измеряет качество вероятностных предсказаний — это и есть кросс-энтропия, усреднённая по всем наблюдениям:

$$
\text{Log-loss} = -\frac{1}{n} \sum_{i=1}^{n} \bigl[ y_i \log p_i + (1 - y_i) \log (1 - p_i) \bigr].
$$

В отличие от accuracy, log-loss строго штрафует «уверенные» ошибки: если модель с вероятностью 0.99 предсказывает не тот класс, потери будут очень большими. Именно эта функция оптимизируется при обучении логистической регрессии.




In [ ]:
y_pred = model.predict(X_test_aug)
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Malignant', 'Benign'],
            yticklabels=['Malignant', 'Benign'])
plt.xlabel('Предсказанный класс')
plt.ylabel('Истинный класс')
plt.title('Confusion Matrix')
plt.show()

print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred):.4f}")
print(f"F1-score : {f1_score(y_test, y_pred):.4f}")

y_prob = model.predict_proba(X_test_aug)
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(8,6))
plt.plot(fpr, tpr, label=f'ROC (AUC = {auc:.3f})')
plt.plot([0,1], [0,1], 'k--', label='Случайная модель')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC-кривая')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f"Log-loss: {log_loss(y_test, y_prob):.4f}")



> **Углублённый взгляд: ручной мини-расчёт ROC и AUC**
>
> Чтобы почувствовать метрику, вычислим её вручную для маленького набора из 6 объектов. Предположим, обученная модель выдала следующие вероятности (для задачи «спам / не спам»):
>
> | $i$ | $p_i$ (спам) | Истина $y$ |
> |------|----------------|--------------|
> | 1 | 0.9 | 1 |
> | 2 | 0.8 | 0 |
> | 3 | 0.6 | 1 |
> | 4 | 0.4 | 0 |
> | 5 | 0.3 | 1 |
> | 6 | 0.2 | 0 |
>
> Отсортируем объекты по убыванию $p_i$ и будем варьировать порог. Получаем точки ROC:
> - Порог $+\infty$: TPR=0, FPR=0.
> - Порог 0.85: объект 1 — TP, TPR=1/3, FPR=0.
> - Порог 0.7: добавляется объект 2 (FP), TPR=1/3, FPR=1/3.
> - Порог 0.5: добавляется объект 3 (TP), TPR=2/3, FPR=1/3.
> - Порог 0.35: добавляется объект 4 (FP), TPR=2/3, FPR=2/3.
> - Порог 0.25: добавляется объект 5 (TP), TPR=1, FPR=2/3.
> - Порог $-\infty$: TPR=1, FPR=1.
>
> Соединив точки, получаем ступенчатую кривую. Площадь под ней (AUC) можно вычислить как вероятность правильного попарного ранжирования: всего пар положительный–отрицательный $3 \times 3 = 9$. Объект 1 (0.9) выше отрицательных 2,4,6 (0.8,0.4,0.2) — 2 победы. Объект 3 (0.6) выше 4 (0.4) и 6 (0.2) — 2 победы. Объект 5 (0.3) выше только 6 (0.2) — 1 победа. Итого 5 побед, AUC = 5/9 ≈ 0.556. Такое значение говорит о слабой разделяющей способности модели на этих данных. Этот расчёт раскрывает вероятностную природу AUC: она оценивает, насколько хорошо модель ранжирует объекты.

# 8. Выбор порога и стоимость ошибок

Порог по умолчанию (0.5) далеко не всегда оптимален. В задаче кредитного скоринга пропуск дефолта (FN) обходится банку гораздо дороже, чем отказ добросовестному клиенту (FP). Пусть матрица стоимостей такова: убыток от дефолта — 10 000 у.е., убыток от отказа хорошему заёмщику — 1 000 у.е., а верные решения не приносят потерь. Тогда ожидаемая стоимость при пороге $t$ составит $10\,000 \cdot FN + 1\,000 \cdot FP$. Варьируя порог, можно найти минимум этой величины.

На практике строят кривые precision, recall и F1 как функции порога и выбирают порог, максимизирующий целевую метрику.




In [ ]:
probs = model.predict_proba(X_test_aug)
precisions, recalls, thresholds_pr, f1s = [], [], [], []
for t in np.linspace(0.01, 0.99, 100):
    preds = (probs >= t).astype(int)
    precisions.append(precision_score(y_test, preds, zero_division=0))
    recalls.append(recall_score(y_test, preds))
    f1s.append(f1_score(y_test, preds, zero_division=0))
    thresholds_pr.append(t)

plt.figure(figsize=(8,5))
plt.plot(thresholds_pr, precisions, label='Precision')
plt.plot(thresholds_pr, recalls, label='Recall')
plt.plot(thresholds_pr, f1s, label='F1')
plt.xlabel('Порог')
plt.ylabel('Значение метрики')
plt.title('Зависимость метрик от порога')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

best_t = thresholds_pr[np.argmax(f1s)]
print(f"Оптимальный порог по F1: {best_t:.3f}")


Аналогичным образом можно написать функцию, минимизирующую произвольную функцию стоимости, заданную пользователем:

In [ ]:
def optimal_threshold_cost(y_true, y_prob, cost_FP, cost_FN):
    best_cost = np.inf
    best_t = 0.5
    for t in np.linspace(0.01, 0.99, 200):
        preds = (y_prob >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
        total_cost = cost_FP * fp + cost_FN * fn
        if total_cost < best_cost:
            best_cost = total_cost
            best_t = t
    return best_t, best_cost

t_opt, cost_opt = optimal_threshold_cost(y_test, y_prob, cost_FP=1000, cost_FN=10000)
print(f"Оптимальный порог по стоимости: {t_opt:.3f} (стоимость {cost_opt:.0f} у.е.)")


Итак, логистическая регрессия не только даёт вероятности, но и позволяет гибко настраивать решающее правило под конкретные бизнес-требования. Следующий шаг — добавление регуляризации (L1, L2, Elastic Net) для борьбы с переобучением и отбора признаков, чему будет посвящена следующая часть главы.

---

## Вопросы для самопроверки

### Для начинающих
1. Почему для обучения логистической регрессии используют градиентный спуск, а не аналитическое решение, как в линейной регрессии?
2. Что такое кросс-энтропийная функция потерь и как она выглядит? Почему она подходит для вероятностных предсказаний?
3. Объясните различие между precision и recall. В каких задачах важнее один, а в каких — другой показатель?
4. Что показывает ROC-кривая и как интерпретировать AUC? Может ли AUC быть меньше 0.5?
5. Зачем и как можно менять порог классификации? Приведите практический пример.

### Для профессионалов
1. Выведите градиент кросс-энтропии в матричной форме и сравните его с градиентом MSE. В чём ключевое различие с точки зрения линейности по $\mathbf{w}$?
2. Опишите метод IRLS. Почему каждая итерация эквивалентна решению взвешенной задачи наименьших квадратов, и в чём преимущество такого подхода перед обычным градиентным спуском?
3. Как работает стохастический градиентный спуск для логистической регрессии? Почему, несмотря на шум, он сходится? Какие расписания скорости обучения гарантируют сходимость согласно условиям Роббинса–Монро?
4. Каким образом AUC связана с вероятностью правильного ранжирования пары объектов разных классов? Докажите это утверждение.
5. Предложите стратегию выбора порога классификации, если известна матрица стоимостей ошибок. Как формально записать задачу минимизации ожидаемых потерь?



# 9. Асимптотические свойства оценки максимального правдоподобия

Оценка $\hat{\mathbf{w}}$, доставляющая максимум логарифмической функции правдоподобия $\ell(\mathbf{w})$, наследует все желаемые асимптотические свойства, которыми обладает метод максимального правдоподобия (MLE) в регулярных статистических моделях. Логистическая регрессия удовлетворяет стандартным условиям регулярности: параметрическое пространство $\mathbb{R}^M$ открыто, истинный вектор параметров $\mathbf{w}^*$ является внутренней точкой, логарифм правдоподобия трижды дифференцируем, а информационная матрица положительно определена. Поэтому для $\hat{\mathbf{w}}$ справедливы следующие утверждения.

**Информационная матрица Фишера.**  
Для одного наблюдения $(\mathbf{x}, y)$ логарифм правдоподобия равен
$$
\ell_i(\mathbf{w}) = y \log p + (1-y)\log(1-p), \quad p = \sigma(\mathbf{w}^T\boldsymbol{\phi}(\mathbf{x})).
$$
Вторая производная по компонентам $w_j, w_k$ (элемент гессиана с обратным знаком) имеет вид
$$
-\frac{\partial^2 \ell_i}{\partial w_j \partial w_k} = p(1-p)\,\phi_j(\mathbf{x})\phi_k(\mathbf{x}).
$$
Этот результат не зависит от реализации $y$, поэтому наблюдаемая информация совпадает со своим математическим ожиданием. Суммируя по всем $n$ наблюдениям, получаем
$$
-\frac{\partial^2 \ell}{\partial \mathbf{w}\partial\mathbf{w}^T} = \Phi^T D \Phi, \quad D = \operatorname{diag}\bigl(p_i(1-p_i)\bigr).
$$
Следовательно, информационная матрица Фишера равна
$$
I(\mathbf{w}) = \mathbb{E}\!\left[-\frac{\partial^2 \ell}{\partial \mathbf{w}\partial\mathbf{w}^T}\right] = \Phi^T D \Phi = H(\mathbf{w}),
$$
где $H(\mathbf{w})$ — гессиан отрицательного логарифма правдоподобия. Совпадение наблюдаемой и ожидаемой информации является прямым следствием того, что логит-функция связи канонична для биномиального распределения. Это свойство упрощает как вычисления, так и теоретический анализ.

**Состоятельность.** При $n \to \infty$ оценка $\hat{\mathbf{w}}$ сходится по вероятности к истинному значению $\mathbf{w}^*$:
$$
\hat{\mathbf{w}} \xrightarrow{p} \mathbf{w}^*.
$$

**Асимптотическая нормальность.** Нормированная ошибка оценки распределена асимптотически нормально:
$$
\sqrt{n}\bigl(\hat{\mathbf{w}} - \mathbf{w}^*\bigr) \xrightarrow{d} \mathcal{N}\!\left(\mathbf{0},\, I_1(\mathbf{w}^*)^{-1}\right),
$$
где $I_1(\mathbf{w}^*) = \lim_{n\to\infty} \frac{1}{n} I(\mathbf{w}^*)$ — предельная средняя информация на одно наблюдение. На практике ковариационную матрицу $\hat{\mathbf{w}}$ оценивают как $(\Phi^T \hat{D} \Phi)^{-1}$ с $\hat{D} = \operatorname{diag}(\hat{p}_i(1-\hat{p}_i))$. Этот результат позволяет строить асимптотические доверительные интервалы для отдельных коэффициентов и проверять гипотезы с помощью теста Вальда.

**Эффективность.** Оценка $\hat{\mathbf{w}}$ достигает границы Крамера–Рао: для любого несмещённого estimator'а $\tilde{\mathbf{w}}$ ковариационная матрица удовлетворяет неравенству $\operatorname{Var}(\tilde{\mathbf{w}}) \succeq I(\mathbf{w})^{-1}$ (в смысле Лёвнера). Таким образом, MLE асимптотически оптимален среди всех состоятельных и асимптотически нормальных оценок.

> **Углублённый взгляд: вывод асимптотической нормальности.**  
> Рассмотрим разложение градиента логарифмической функции правдоподобия в точке $\hat{\mathbf{w}}$ вокруг $\mathbf{w}^*$. Поскольку $\nabla \ell(\hat{\mathbf{w}}) = \mathbf{0}$, имеем
> $$
> \mathbf{0} = \nabla \ell(\mathbf{w}^*) + H(\bar{\mathbf{w}})(\hat{\mathbf{w}} - \mathbf{w}^*),
> $$
> где $\bar{\mathbf{w}}$ лежит между $\hat{\mathbf{w}}$ и $\mathbf{w}^*$, а $H(\bar{\mathbf{w}}) = -\Phi^T \bar{D} \Phi$ — гессиан в промежуточной точке. Тогда
> $$
> \sqrt{n}(\hat{\mathbf{w}} - \mathbf{w}^*) = -\left(\frac{1}{n} H(\bar{\mathbf{w}})\right)^{-1} \frac{1}{\sqrt{n}} \nabla \ell(\mathbf{w}^*).
> $$
> Градиент $\nabla \ell(\mathbf{w}^*) = \Phi^T(\mathbf{y} - \mathbf{p}^*)$ есть сумма $n$ независимых случайных векторов с нулевым средним и ковариационной матрицей $\Phi_i \Phi_i^T p_i^*(1-p_i^*)$, т.е. в точности $I(\mathbf{w}^*)$. По центральной предельной теореме $\frac{1}{\sqrt{n}} \nabla \ell(\mathbf{w}^*) \xrightarrow{d} \mathcal{N}(\mathbf{0}, I_1(\mathbf{w}^*))$. В силу состоятельности $\frac{1}{n} H(\bar{\mathbf{w}}) \xrightarrow{p} I_1(\mathbf{w}^*)$. Применяя теорему Слуцкого, получаем требуемое утверждение.

# 10. Тест отношения правдоподобия (LRT)

Для сравнения вложенных моделей аппарат максимального правдоподобия предоставляет универсальный инструмент — **тест отношения правдоподобия** (Likelihood Ratio Test, LRT). Пусть модель $M_0$ является частным случаем модели $M_1$, т.е. $M_0$ получается наложением $k$ линейных ограничений на вектор параметров $\mathbf{w}$ (например, обнулением $k$ коэффициентов). Обозначим через $\hat{\mathbf{w}}_0$ и $\hat{\mathbf{w}}_1$ оценки MLE для этих моделей, а $\ell(\hat{\mathbf{w}}_0)$ и $\ell(\hat{\mathbf{w}}_1)$ — соответствующие максимумы логарифма правдоподобия. Статистика LRT определяется как
$$
\lambda = 2 \bigl( \ell(\hat{\mathbf{w}}_1) - \ell(\hat{\mathbf{w}}_0) \bigr).
$$
При нулевой гипотезе $H_0: \mathbf{w} \in M_0$ величина $\lambda$ асимптотически распределена как $\chi^2$ с числом степеней свободы $k$, равным разности размерностей моделей. Интуитивно: если ограничения справедливы, добавление лишних признаков не должно значительно увеличивать правдоподобие, и $\lambda$ будет малым.

Классический пример — проверка значимости целой группы признаков. Пусть полная модель включает $D$ предикторов, а редуцированная — лишь первые $d < D$. Тогда
$$
\lambda = 2 \left( \ell(\hat{\mathbf{w}}_{\text{full}}) - \ell(\hat{\mathbf{w}}_{\text{reduced}}) \right) \xrightarrow{d} \chi^2_{D-d}.
$$
Большое значение статистики свидетельствует о том, что исключённые признаки совместно улучшают качество модели.

> **Углублённый взгляд: вывод асимптотики LRT.**  
> Разложим $\ell(\hat{\mathbf{w}}_0)$ в ряд Тейлора вокруг $\hat{\mathbf{w}}_1$. Поскольку $\nabla \ell(\hat{\mathbf{w}}_1) = \mathbf{0}$, имеем
> $$
> \ell(\hat{\mathbf{w}}_0) \approx \ell(\hat{\mathbf{w}}_1) - \frac{1}{2} (\hat{\mathbf{w}}_0 - \hat{\mathbf{w}}_1)^T I(\hat{\mathbf{w}}_1) (\hat{\mathbf{w}}_0 - \hat{\mathbf{w}}_1).
> $$
> Разность $\hat{\mathbf{w}}_0 - \hat{\mathbf{w}}_1$ при нулевой гипотезе асимптотически нормальна с нулевым средним и ковариационной матрицей, связанной с проектором на ограничения. Подстановка приводит к квадратичной форме от асимптотически нормального вектора, которая и даёт $\chi^2$ распределение. Строгое доказательство опирается на теорему Уилкса.

# 11. Логистическая регрессия как обобщённая линейная модель (GLM)

Логистическая регрессия естественно вписывается в единую структуру **обобщённых линейных моделей** (Generalized Linear Models, GLM), предложенную Nelder и Wedderburn (1972). GLM состоит из трёх компонентов:

1. **Случайная компонента**: отклик $y$ имеет распределение из экспоненциального семейства. Плотность (или вероятность) может быть записана в каноническом виде
   $$
   f(y; \theta, \phi) = \exp\left( \frac{y\theta - b(\theta)}{a(\phi)} + c(y,\phi) \right),
   $$
   где $\theta$ — канонический параметр, $\phi$ — параметр масштаба (дисперсии). Для биномиального распределения $y \sim \text{Binomial}(1, p)$:
   $$
   \theta = \log\frac{p}{1-p} = \text{logit}(p), \quad b(\theta) = \log(1+e^\theta), \quad a(\phi) = 1.
   $$

2. **Систематическая компонента**: линейный предиктор $\eta = \mathbf{w}^T \boldsymbol{\phi}(\mathbf{x})$.

3. **Функция связи** $g(\cdot)$, связывающая математическое ожидание $\mu = \mathbb{E}[y]$ с линейным предиктором: $g(\mu) = \eta$. Для логистической регрессии $g$ — логит: $\eta = \log\frac{\mu}{1-\mu}$.

Если функция связи выбрана так, что $\eta = \theta$, т.е. линейный предиктор напрямую моделирует канонический параметр, говорят о **канонической связи**. Именно логит является канонической связью для биномиального распределения. Каноническая связь даёт два ключевых преимущества:

- Достаточной статистикой для $\mathbf{w}$ является $\Phi^T \mathbf{y}$, что упрощает как алгоритмическую реализацию, так и теоретический анализ.
- Наблюдаемая и ожидаемая информационные матрицы Фишера совпадают: $-\partial^2 \ell / \partial \mathbf{w}\partial\mathbf{w}^T = I(\mathbf{w})$. Это гарантирует, что оптимизация метода Ньютона–Рафсона эквивалентна итеративному взвешенному МНК (IRLS) без необходимости вычислять ожидание гессиана.

> **Расширение: Пуассоновская регрессия**
>
> Для откликов, представляющих собой счётные данные (число событий), биномиальное распределение заменяется пуассоновским. Пуассоновская регрессия — это GLM с распределением $y \sim \text{Poisson}(\lambda)$. Плотность пуассоновского распределения в экспоненциальной форме:
> $$
> f(y;\lambda) = \exp\bigl(y\log\lambda - \lambda - \log y!\bigr),
> $$
> откуда $\theta = \log\lambda$, $b(\theta) = e^\theta$. Каноническая функция связи — логарифмическая: $\eta = \log\lambda$, так что $\lambda = \exp(\mathbf{w}^T \boldsymbol{\phi}(\mathbf{x}))$. Коэффициенты интерпретируются мультипликативно: увеличение $\phi_j$ на единицу умножает ожидаемое количество событий на $e^{w_j}$.
>
> Сгенерируем данные о количестве дорожно-транспортных происшествий в зависимости от интенсивности движения `traffic` и погодных условий `weather` (0 — ясно, 1 — осадки).



In [ ]:
np.random.seed(42)
n = 200
traffic = np.random.uniform(0.5, 2.0, n)
weather = np.random.binomial(1, 0.4, n)
log_lambda = 0.5 + 1.2 * traffic + 0.8 * weather
lam = np.exp(log_lambda)
accidents = np.random.poisson(lam)
#Обучим Пуассоновскую регрессию с помощью `statsmodels`:

import statsmodels.api as sm
X = sm.add_constant(np.column_stack((traffic, weather)))
model_pois = sm.GLM(accidents, X, family=sm.families.Poisson()).fit()
print(model_pois.summary())


> В выдаче мы увидим оценки коэффициентов, близкие к истинным $(0.5, 1.2, 0.8)$. Их интерпретация: увеличение `traffic` на 1 единицу повышает ожидаемое число ДТП в $e^{1.2} \approx 3.32$ раза, а осадки — в $e^{0.8} \approx 2.23$ раза при фиксированном трафике.

# 12. Вероятностная калибровка

Логистическая регрессия возвращает числа от 0 до 1, которые мы интерпретируем как вероятности. Однако оптимизация кросс-энтропии сама по себе не гарантирует, что эти числа будут **хорошо откалиброваны**, т.е. что среди объектов с предсказанной вероятностью $p$ доля положительных исходов действительно равна $p$. Смещение калибровки может возникать из-за неправильной спецификации модели, регуляризации или дисбаланса классов.

Основной метрикой калибровки служит **Brier score**:
$$
\text{BS} = \frac{1}{n}\sum_{i=1}^{n} (p_i - y_i)^2,
$$
который принимает значения от 0 (идеальная калибровка и точность) до 0.25 (максимальная неопределённость при $p_i \equiv 0.5$). Brier score можно разложить на компоненты refinement (разрешающая способность) и calibration (надёжность), но на практике чаще используют калибровочные кривые.

**Калибровочная кривая** (reliability diagram) строится разбиением предсказанных вероятностей на бины (например, 10 интервалов равной ширины). Для каждого бина вычисляется средняя предсказанная вероятность $\bar{p}_k$ и эмпирическая частота положительных исходов $\bar{y}_k$. В хорошо откалиброванной модели точки $(\bar{p}_k, \bar{y}_k)$ лежат на диагонали.

Если калибровка нарушена, применяют **постобработку** предсказаний:

- **Platt scaling** подгоняет одномерную логистическую регрессию к выходным логитам модели: $\hat{p}_{\text{cal}} = \sigma(a z + b)$, где $z = \mathbf{w}^T \boldsymbol{\phi}(\mathbf{x})$ — исходный логит, а $a, b$ настраиваются на валидационной выборке.
- **Изотоническая регрессия** решает задачу $\min_{\hat{p}} \sum (y_i - \hat{p}_i)^2$ при ограничении $\hat{p}_1 \le \dots \le \hat{p}_m$ (монотонность). Она непараметрична и может исправить любую монотонную деформацию, но требует больше данных.

Продемонстрируем калибровку на модели с сильной L1-регуляризацией, которая намеренно ухудшает вероятностные оценки.



In [ ]:
from sklearn.calibration import CalibratedClassifierCV, calibration_curve

# Модель с L1-регуляризацией (плохая калибровка)
model_l1 = LogisticRegression(penalty='l1', solver='saga', C=0.05, max_iter=5000)
model_l1.fit(X_train_scaled, y_train)
y_prob_l1 = model_l1.predict_proba(X_test_scaled)[:, 1]

# Изотоническая калибровка на кросс-валидации
calibrated = CalibratedClassifierCV(
    estimator=LogisticRegression(penalty='l1', solver='saga', C=0.05, max_iter=5000),
    method='isotonic', cv=5
)
calibrated.fit(X_train_scaled, y_train)
y_prob_cal = calibrated.predict_proba(X_test_scaled)[:, 1]

# Построение калибровочных кривых
plt.figure(figsize=(8,6))
for probs, label in [(y_prob_l1, 'До калибровки'), (y_prob_cal, 'После изотонической калибровки')]:
    prob_true, prob_pred = calibration_curve(y_test, probs, n_bins=10)
    plt.plot(prob_pred, prob_true, marker='o', label=label)
plt.plot([0,1],[0,1], 'k--', label='Идеал')
plt.xlabel('Предсказанная вероятность')
plt.ylabel('Эмпирическая частота')
plt.title('Калибровочные кривые')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


График наглядно показывает, как L1-регуляризация искажает калибровку (кривая отклоняется от диагонали), а изотоническая регрессия возвращает её практически на место.

# 13. Взвешенная логистическая регрессия для несбалансированных классов

При значительном дисбалансе классов (например, 1% положительных и 99% отрицательных) стандартная кросс-энтропия толкает модель к тривиальному предсказанию «всегда ноль», поскольку это даёт низкий log-loss. Для борьбы с этим явлением вводят **взвешенную логистическую регрессию**, назначая разные штрафы за ошибки на разных классах. Взвешенная функция потерь имеет вид
$$
E_{\text{w}}(\mathbf{w}) = -\sum_{i=1}^{n} \Bigl[ c_1 y_i \log p_i + c_0 (1-y_i) \log(1-p_i) \Bigr],
$$
где $c_1, c_0 > 0$ — веса, обычно выбираемые обратно пропорциональными частотам классов или на основе матрицы стоимостей.

Выведем градиент $E_{\text{w}}$. Частная производная по $w_k$:
$$
\frac{\partial E_{\text{w}}}{\partial w_k} = -\sum_{i} \left[ c_1 y_i \frac{1}{p_i} \frac{\partial p_i}{\partial w_k} + c_0 (1-y_i) \frac{1}{1-p_i} \left(-\frac{\partial p_i}{\partial w_k}\right) \right].
$$
Подставляя $\frac{\partial p_i}{\partial w_k} = p_i(1-p_i)\phi_k(\mathbf{x}_i)$, получаем
$$
\frac{\partial E_{\text{w}}}{\partial w_k} = -\sum_{i} \Bigl[ c_1 y_i (1-p_i) - c_0 (1-y_i) p_i \Bigr] \phi_k(\mathbf{x}_i).
$$
Внутри скобок после раскрытия: $c_1 y_i - c_1 y_i p_i - c_0 p_i + c_0 y_i p_i = c_1 y_i - c_0 p_i + y_i p_i (c_0 - c_1)$. Это выражение можно перегруппировать в виде $v_i (p_i - y_i)$, если положить
$$
v_i =
\begin{cases}
c_1, & y_i = 1,\\
c_0, & y_i = 0.
\end{cases}
$$
Проверим: для $y_i=1$ имеем $v_i(p_i - 1) = c_1(p_i - 1) = -c_1(1-p_i)$, что совпадает с $-[c_1 \cdot 1 \cdot (1-p_i) - c_0 \cdot 0 \cdot p_i]$. Для $y_i=0$: $v_i(p_i - 0) = c_0 p_i$, а в скобках стоит $-[c_1 \cdot 0 \cdot (1-p_i) - c_0 \cdot 1 \cdot p_i] = c_0 p_i$. Знак минус перед суммой в исходном выражении компенсируется, и окончательно
$$
\frac{\partial E_{\text{w}}}{\partial w_k} = \sum_{i=1}^{n} v_i (p_i - y_i) \phi_k(\mathbf{x}_i).
$$
В матричной форме
$$
\nabla_{\mathbf{w}} E_{\text{w}} = \Phi^T \mathbf{v} \circ (\mathbf{p} - \mathbf{y}),
$$
где $\circ$ — поэлементное умножение, а вектор $\mathbf{v}$ составлен из весов $v_i$. Таким образом, градиент получается из стандартного домножением каждой невязки $(p_i - y_i)$ на вес $v_i$, соответствующий истинному классу наблюдения.

В библиотеке `sklearn` взвешенная логистическая регрессия реализована через параметр `class_weight='balanced'`, который автоматически устанавливает $c_1 = \frac{n}{2n_1}$, $c_0 = \frac{n}{2n_0}$, где $n_1, n_0$ — количество объектов каждого класса в обучающей выборке. Это делает вклад каждого класса в функцию потерь примерно равным.



In [ ]:
model_balanced = LogisticRegression(penalty=None, solver='lbfgs', max_iter=10000,
                                    class_weight='balanced')
model_balanced.fit(X_train_scaled, y_train)
y_pred_bal = model_balanced.predict(X_test_scaled)

print("Взвешенная модель:")
print(f"  Precision: {precision_score(y_test, y_pred_bal):.4f}")
print(f"  Recall   : {recall_score(y_test, y_pred_bal):.4f}")
print(f"  F1-score : {f1_score(y_test, y_pred_bal):.4f}")


Сравнение с невзвешенной моделью показывает рост полноты (recall) для миноритарного класса, обычно сопровождаемый некоторым снижением точности (precision). Это — прямой результат того, что штраф за пропуск положительного объекта стал выше. Взвешенная логистическая регрессия сохраняет всю интерпретируемость базовой модели и остаётся одним из наиболее прозрачных методов работы с несбалансированными данными.

# 14. Регуляризация в логистической регрессии

Регуляризация в логистической регрессии преследует те же цели, что и в линейной: уменьшить переобучение, повысить численную устойчивость и, в случае $L_1$-штрафа, выполнить автоматический отбор признаков. Добавка к кросс-энтропии штрафного члена, пропорционального норме вектора весов, ограничивает свободу модели и заставляет коэффициенты стягиваться к нулю.

**$L_2$-регуляризация (Ridge)** добавляет квадратичный штраф:
$$
E_{\text{ridge}}(\mathbf{w}) = -\sum_{i=1}^{n} \bigl[y_i \log p_i + (1-y_i)\log(1-p_i)\bigr] + \frac{\lambda}{2} \sum_{j=1}^{M-1} w_j^2,
$$
где свободный член $w_0$ не штрафуется. Градиент модифицируется добавкой $\lambda \mathbf{w}$ (кроме $w_0$): $\nabla E_{\text{ridge}} = \Phi^T(\mathbf{p} - \mathbf{y}) + \lambda \mathbf{w}_{\text{no bias}}$. В библиотеке `sklearn` сила регуляризации задаётся параметром $C = 1/\lambda$: чем меньше $C$, тем сильнее штраф.



> **Связь $\lambda$ и $C$ в sklearn.**  
> В библиотеке `sklearn` параметр регуляризации обозначается $C$ и связан с $\lambda$ соотношением $C = 1/\lambda$.  
> - **$C \to \infty$** ($\lambda \to 0$) — регуляризация исчезает, модель стремится к обычной логистической регрессии.
> - **$C \to 0$** ($\lambda \to \infty$) — регуляризация максимальна, все коэффициенты (кроме $w_0$) стремятся к нулю.
>
> На практике $C$ подбирают по сетке (например, `np.logspace(-3, 3, 20)`) с помощью кросс-валидации.


**$L_1$-регуляризация (Lasso)** добавляет сумму абсолютных значений:
$$
E_{\text{lasso}}(\mathbf{w}) = -\sum_{i=1}^{n} \bigl[y_i \log p_i + (1-y_i)\log(1-p_i)\bigr] + \lambda \sum_{j=1}^{M-1} |w_j|.
$$
Недифференцируемость модуля в нуле требует использования субградиентных методов. Субградиент функции $|w_j|$ — это любое число из $[-1,1]$ при $w_j = 0$, $\operatorname{sign}(w_j)$ при $w_j \neq 0$. Условие оптимальности $\mathbf{0} \in \partial E_{\text{lasso}}$ приводит к тому, что часть коэффициентов оказывается **точно равной нулю**. Геометрически это объясняется формой области допустимых значений штрафа: $L_1$-шар (ромб) имеет острые вершины на осях, куда «притягивается» решение при ограничении. $L_2$-шар (круг) же гладкий и обычно даёт ненулевые, но сжатые коэффициенты.

> **Углублённый взгляд: координатный спуск и мягкий порог**
>
> При координатном спуске каждая переменная $w_j$ обновляется при фиксированных остальных. Зафиксируем все веса, кроме $w_j$, и подставим $p_i = \sigma(\mathbf{w}_{-j}^T \boldsymbol{\phi}(\mathbf{x}_i) + w_j \phi_j(\mathbf{x}_i))$. Целевая функция становится одномерной, и можно найти её минимум аналитически, используя суррогатное квадратичное приближение. В результате обновление принимает форму оператора мягкого порога (soft-thresholding):
> $$
> w_j := S_{\lambda/\|\boldsymbol{\phi}_j\|^2}\!\left( \frac{1}{\|\boldsymbol{\phi}_j\|^2}[\Phi^T(\mathbf{y} - \mathbf{p}_{-j})]_j \right), \qquad S_\lambda(x) = \operatorname{sign}(x)\max(|x|-\lambda, 0).
> $$
> Здесь $\mathbf{p}_{-j}$ — вектор вероятностей без учёта $j$-го признака. Оператор $S_\lambda$ зануляет $x$, если его абсолютное значение меньше $\lambda$, и уменьшает на $\lambda$ в противном случае. Таким образом, малые по модулю коэффициенты автоматически обнуляются.

**Elastic Net** комбинирует оба штрафа с параметром смешивания $\rho$: $\lambda \bigl(\rho \sum |w_j| + \frac{1-\rho}{2} \sum w_j^2\bigr)$. Он сохраняет разреженность $L_1$ и стабильность $L_2$ при коррелированных признаках.



Оптимальная модель оставляет лишь часть признаков, автоматически исключая шумовые и некоторые исходные, что улучшает интерпретируемость и обобщающую способность.



**Демонстрация на данных breast cancer с шумовыми признаками.**  
Добавим к исходным 30 признакам 50 случайных шумовых, чтобы увидеть отбор признаков. Подберём $C$ и $l_1$-ratio с помощью кросс-валидации.

> **Почему `solver='saga'`?**  
> Для L1- и Elastic Net-регуляризации в логистической регрессии `sklearn` требует `solver='saga'` (или `'liblinear'` для L1). `'saga'` поддерживает все типы регуляризации (L1, L2, Elastic Net), хорошо масштабируется на большие данные и работает с `penalty='elasticnet'` и параметром `l1_ratio`.




In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegressionCV, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

# ============================================================
# 1. Загрузка и подготовка данных
# ============================================================
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Добавление 50 шумовых признаков
np.random.seed(0)
noise = np.random.randn(X_train_scaled.shape[0], 50)
X_train_noisy = np.hstack([X_train_scaled, noise])
X_test_noisy = np.hstack([X_test_scaled, np.random.randn(X_test_scaled.shape[0], 50)])

print(f"Размерность данных: {X_train_noisy.shape[1]} признаков")
print(f"  - Исходных: {X_train_scaled.shape[1]}")
print(f"  - Шумовых: 50")
print("-" * 50)

# ============================================================
# 2. Подбор параметров Elastic Net через кросс-валидацию
# ============================================================
cv_model = LogisticRegressionCV(
    penalty='elasticnet',
    solver='saga',
    Cs=10,
    l1_ratios=[0.1, 0.5, 0.7, 0.9, 1.0],
    cv=5,
    max_iter=10000,
    random_state=42,
    n_jobs=-1
)
cv_model.fit(X_train_noisy, y_train)

print(f"\nОптимальные параметры Elastic Net:")
print(f"  C = {cv_model.C_[0]:.4f} (чем меньше, тем сильнее регуляризация)")
print(f"  l1_ratio = {cv_model.l1_ratio_[0]:.3f} (1.0 = чистый L1, 0 = чистый L2)")
print(f"  Точно нулевых коэффициентов: {np.sum(np.abs(cv_model.coef_) < 1e-10)}")
print("-" * 50)

# ============================================================
# 3. L1-регуляризационный путь (как меняются коэффициенты)
# ============================================================
Cs = np.logspace(-3, 1, 30)
coef_l1 = []

for C in Cs:
    lr = LogisticRegression(
        penalty='l1',
        C=C,
        solver='saga',
        max_iter=10000,
        random_state=42
    )
    lr.fit(X_train_noisy, y_train)
    coef_l1.append(lr.coef_.ravel())

coef_l1 = np.array(coef_l1)

# ============================================================
# 4. L2-регуляризационный путь для сравнения
# ============================================================
coef_l2 = []

for C in Cs:
    lr = LogisticRegression(
        penalty='l2',
        C=C,
        solver='lbfgs',
        max_iter=10000,
        random_state=42
    )
    lr.fit(X_train_noisy, y_train)
    coef_l2.append(lr.coef_.ravel())

coef_l2 = np.array(coef_l2)

# ============================================================
# 5. Визуализация: сравнение L1 и L2
# ============================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# L1 (Lasso) — разреженный
for i in range(coef_l1.shape[1]):
    ax1.semilogx(Cs, coef_l1[:, i], lw=0.8, alpha=0.5, color='steelblue')
ax1.axvline(cv_model.C_[0], color='red', linestyle='--', linewidth=2,
            label=f'Оптимальный C = {cv_model.C_[0]:.4f}')
ax1.set_xlabel('C (обратный λ)', fontsize=12)
ax1.set_ylabel('Коэффициенты', fontsize=12)
ax1.set_title('L1-регуляризация (Lasso) — обнуляет коэффициенты', fontsize=12)
ax1.legend()
ax1.grid(alpha=0.3)

# L2 (Ridge) — гладкое сжатие
for i in range(coef_l2.shape[1]):
    ax2.semilogx(Cs, coef_l2[:, i], lw=0.8, alpha=0.5, color='steelblue')
ax2.axvline(cv_model.C_[0], color='red', linestyle='--', linewidth=2,
            label=f'Оптимальный C = {cv_model.C_[0]:.4f}')
ax2.set_xlabel('C (обратный λ)', fontsize=12)
ax2.set_ylabel('Коэффициенты', fontsize=12)
ax2.set_title('L2-регуляризация (Ridge) — сжимает, но не обнуляет', fontsize=12)
ax2.legend()
ax2.grid(alpha=0.3)

plt.suptitle('Сравнение регуляризационных путей логистической регрессии', fontsize=14)
plt.tight_layout()
plt.show()

# ============================================================
# 6. Отдельный график: траектория одного коэффициента
# ============================================================
# Выбираем один из исходных признаков (например, первый)
feature_idx = 0
coef_l1_selected = [coef_l1[i, feature_idx] for i in range(len(Cs))]
coef_l2_selected = [coef_l2[i, feature_idx] for i in range(len(Cs))]

plt.figure(figsize=(10, 5))
plt.semilogx(Cs, coef_l1_selected, 'o-', linewidth=2, markersize=4,
             label=f'L1 (признак {feature_idx})', color='blue')
plt.semilogx(Cs, coef_l2_selected, 's-', linewidth=2, markersize=4,
             label=f'L2 (признак {feature_idx})', color='red')
plt.axvline(cv_model.C_[0], color='green', linestyle='--', linewidth=2,
            label=f'Оптимальный C = {cv_model.C_[0]:.4f}')
plt.xlabel('C (обратный λ)', fontsize=12)
plt.ylabel('Значение коэффициента', fontsize=12)
plt.title(f'Сравнение L1 и L2 регуляризации для одного признака', fontsize=12)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================================
# 7. Итоговая интерпретация
# ============================================================
n_nonzero = np.sum(np.abs(cv_model.coef_) > 1e-6)
n_zero = X_train_noisy.shape[1] - n_nonzero

print("\n" + "=" * 50)
print("ИТОГОВАЯ ИНТЕРПРЕТАЦИЯ")
print("=" * 50)
print(f"Всего признаков:               {X_train_noisy.shape[1]}")
print(f"Ненулевых коэффициентов:       {n_nonzero}")
print(f"Обнулённых (отброшенных):      {n_zero}")
print(f"→ L1-регуляризация обнулила {n_zero} признаков из {X_train_noisy.shape[1]}")
print(f"→ Из них шумовых:              {max(0, n_zero - (X_train_scaled.shape[1] - n_nonzero))}")
print("\nВывод: L1-регуляризация успешно отбирает значимые признаки,")
print("автоматически исключая шумовые и некоторые коррелированные исходные.")



В данном примере из 80 признаков (30 исходных + 50 шумовых) модель оставила только **31** ненулевой коэффициент. Это демонстрирует, как L1-регуляризация автоматически отбирает значимые признаки, обнуляя шумовые и некоторые коррелированные исходные.

### Рекомендации по выбору регуляризации

| Тип | Эффект | Когда использовать |
|-----|--------|-------------------|
| **L2 (Ridge)** | Сжимает коэффициенты, но не обнуляет | Много признаков, важны все; мультиколлинеарность |
| **L1 (Lasso)** | Обнуляет коэффициенты (отбор признаков) | Нужна интерпретируемость; много шумовых признаков |
| **Elastic Net** | Комбинация L1 и L2 | Сильно коррелированные признаки; L1 выбирает только один из группы |


**Байесовская интерпретация.**  
Штраф $L_2$ эквивалентен гауссовскому априорному распределению коэффициентов $w_j \sim \mathcal{N}(0, 1/\lambda)$, а $L_1$ — лапласовскому $p(w_j) \propto \exp(-\lambda |w_j|)$. Острый пик лапласовского распределения в нуле объясняет способность $L_1$ порождать разреженные решения.




# 15. Диагностика модели

Даже при хороших метриках на тесте необходимо проверять соблюдение предположений модели. Для логистической регрессии это, в первую очередь, отсутствие мультиколлинеарности, адекватность спецификации и выявление влиятельных наблюдений.

**Мультиколлинеарность.** Коэффициент инфляции дисперсии (VIF) измеряет, насколько дисперсия оценки коэффициента раздута из-за линейной зависимости признаков. Вычисляем VIF и строим тепловую карту корреляций.


In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_const = sm.add_constant(pd.DataFrame(X_train_scaled, columns=data.feature_names))
vif = pd.DataFrame({'feature': X_const.columns, 'VIF': [variance_inflation_factor(X_const.values, i) for i in range(X_const.shape[1])]})
print(vif.sort_values('VIF', ascending=False).head(10))


Тепловая карта корреляций наглядно показывает группы сильно коррелированных признаков (например, `mean radius` и `mean perimeter`).

**Остатки девианса.**  
Вместо обычных остатков $y_i - \hat{p}_i$ в GLM используют остатки девианса:
$$
d_i = \operatorname{sign}(y_i - \hat{p}_i) \sqrt{-2\bigl( y_i \log \hat{p}_i + (1-y_i)\log(1-\hat{p}_i) \bigr)}.
$$
Они симметризируют вклад каждого наблюдения в девианс. График остатков девианса против предсказанных вероятностей или индекса наблюдения помогает выявить выбросы и нелинейность.



In [ ]:
p = model.predict_proba(X_test_aug)

eps = 1e-15
p = np.clip(p, eps, 1 - eps)

deviance_resid = np.where(y_test == 1,
                          np.sqrt(-2 * np.log(p)),
                          -np.sqrt(-2 * np.log(1 - p)))

plt.figure(figsize=(8, 5))
plt.scatter(p, deviance_resid, alpha=0.5)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Предсказанная вероятность')
plt.ylabel('Остатки девианса')
plt.title('График остатков девианса')
plt.grid(alpha=0.3)
plt.show()


**Тест Хосмера–Лемешоу.**  
Разбиваем наблюдения на децили по предсказанной вероятности и сравниваем ожидаемое и наблюдаемое количество положительных исходов в каждой группе. Статистика теста распределена как $\chi^2$ с числом степеней свободы $g-2$, где $g$ — число групп. Реализуем собственную функцию.



In [ ]:
def hosmer_lemeshow(y_true, y_prob, g=10):
    thresholds = np.quantile(y_prob, np.linspace(0, 1, g+1))
    hl_stat = 0
    for i in range(g):
        mask = (y_prob >= thresholds[i]) & (y_prob < thresholds[i+1]) if i < g-1 else (y_prob >= thresholds[i])
        if np.sum(mask) == 0: continue
        obs_pos = y_true[mask].sum()
        exp_pos = y_prob[mask].sum()
        n_g = mask.sum()
        hl_stat += (obs_pos - exp_pos)**2 / (exp_pos * (1 - exp_pos/n_g)) if exp_pos > 0 and exp_pos < n_g else 0
    return hl_stat

hl = hosmer_lemeshow(y_test, p)
print(f"Статистика Хосмера–Лемешоу: {hl:.3f}")


**Расстояние Кука** оценивает влияние каждого наблюдения на вектор коэффициентов. Для логистической регрессии аналог: $D_i = \frac{r_i^2 h_{ii}}{M (1-h_{ii})^2}$, где $r_i$ — стьюдентизированный остаток, $h_{ii}$ — левередж. Приближённо используем влияние из `statsmodels` после подгонки модели на обучающей выборке.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Загрузка и подготовка данных
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Обучаем логистическую регрессию
lr = LogisticRegression(penalty=None, solver='lbfgs', max_iter=1000)
lr.fit(X_train_scaled, y_train)

# --- Приближённое вычисление расстояния Кука ---
# Добавляем столбец единиц
X_with_const = np.hstack([np.ones((X_train_scaled.shape[0], 1)), X_train_scaled])

# Матрица "шляпа" (hat matrix)
H = X_with_const @ np.linalg.pinv(X_with_const.T @ X_with_const) @ X_with_const.T
leverage = np.diag(H)

# Предсказанные вероятности и остатки
p = lr.predict_proba(X_train_scaled)[:, 1]
residuals = y_train - p
mse = np.mean(residuals**2)

# Расстояние Кука (приближённое)
M = X_with_const.shape[1]  # число параметров
cooks_d = (residuals**2 * leverage) / (M * mse * (1 - leverage)**2)

# --- Визуализация ---
plt.figure(figsize=(12, 4))
plt.bar(range(len(cooks_d)), cooks_d, width=0.8, color='steelblue', alpha=0.7)
plt.axhline(y=4 / len(y_train), color='red', linestyle='--',
            label=f'Порог 4/n = {4/len(y_train):.4f}')
plt.xlabel('Номер наблюдения')
plt.ylabel('Расстояние Кука')
plt.title('Расстояние Кука для логистической регрессии (приближённое)')
plt.legend()
plt.grid(alpha=0.3, axis='y')
plt.show()

print(f"Число влиятельных точек (D > 4/n): {np.sum(cooks_d > 4/len(y_train))}")


# 16. Многоклассовая логистическая регрессия

Когда число классов $K > 2$, бинарную модель обобщают двумя основными способами: **One‑vs‑Rest (OvR)** и **Softmax (мультиномиальная)**.

**Softmax-регрессия** напрямую моделирует вероятности всех классов одновременно через обобщённую логистическую функцию:
$$
P(y = k \mid \mathbf{x}) = \frac{\exp(\mathbf{w}_k^T \boldsymbol{\phi}(\mathbf{x}))}{\sum_{j=1}^{K} \exp(\mathbf{w}_j^T \boldsymbol{\phi}(\mathbf{x}))}, \quad k=1,\dots,K.
$$
Для идентифицируемости часто фиксируют $\mathbf{w}_1 = \mathbf{0}$ или налагают условие $\sum \mathbf{w}_k = \mathbf{0}$.

Функция потерь — **категориальная кросс-энтропия**:
$$
E(\mathbf{W}) = -\sum_{i=1}^{n} \sum_{k=1}^{K} \mathbf{1}(y^{(i)} = k) \log \hat{p}_{ik},
$$
где $\hat{p}_{ik} = P(y = k \mid \mathbf{x}^{(i)})$.

> **Углублённый взгляд: вывод градиента для softmax**
>
> Вычислим производную $\hat{p}_{ik}$ по линейному предиктору $z_{ij} = \mathbf{w}_j^T \boldsymbol{\phi}(\mathbf{x}^{(i)})$. Для $j=k$:
> $$
> \frac{\partial \hat{p}_{ik}}{\partial z_{ik}} = \hat{p}_{ik}(1 - \hat{p}_{ik}),
> $$
> для $j \neq k$:
> $$
> \frac{\partial \hat{p}_{ik}}{\partial z_{ij}} = -\hat{p}_{ik}\hat{p}_{ij}.
> $$
> Тогда частная производная функции потерь по $z_{ik}$:
> $$
> \frac{\partial E}{\partial z_{ik}} = \hat{p}_{ik} - \mathbf{1}(y^{(i)}=k).
> $$
> Применяя цепное правило, градиент по вектору $\mathbf{w}_k$:
> $$
> \nabla_{\mathbf{w}_k} E = \sum_{i=1}^{n} (\hat{p}_{ik} - \mathbf{1}(y^{(i)}=k)) \boldsymbol{\phi}(\mathbf{x}^{(i)}).
> $$
> В матричной форме: $\nabla_{\mathbf{W}} E = \Phi^T(\hat{\mathbf{P}} - \mathbf{Y})$, где $\hat{\mathbf{P}}$ — матрица $n \times K$ вероятностей, $\mathbf{Y}$ — one‑hot матрица меток.

**One‑vs‑Rest** обучает $K$ отдельных бинарных классификаторов, каждый из которых отличает свой класс от остальных. Итоговая вероятность принадлежности классу $k$ обычно получается нормировкой выходов сигмоид, но без гарантии, что сумма будет равна 1. Функция потерь — сумма $K$ бинарных кросс-энтропий.

**Сравнительная таблица Softmax vs One‑vs‑Rest**

| Критерий | Softmax (мультиномиальная) | One‑vs‑Rest |
|----------|----------------------------|-------------|
| Выход модели | $K$ вероятностей, сумма = 1 | $K$ вероятностей, сумма может отличаться от 1 |
| Учёт зависимости | Обучается совместно, вероятности конкурируют | Классификаторы независимы |
| Скорость обучения | Медленнее при больших $K$ | Быстрее, можно параллелить |
| Интерпретируемость | Один набор коэффициентов на класс, сложнее | Простая для каждого класса |
| Предпочтение | Когда классы взаимоисключающие и их немного | Большое число классов или перекрывающиеся классы |

Пример на датасете `load_digits` (цифры 0–9).




In [ ]:
from sklearn.datasets import load_digits
digits = load_digits()
X_d, y_d = digits.data, digits.target
X_tr, X_te, y_tr, y_te = train_test_split(X_d, y_d, test_size=0.3, random_state=0)
# Softmax
softmax = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000, C=1.0)
softmax.fit(X_tr, y_tr)
y_pred_sm = softmax.predict(X_te)

print("Softmax accuracy:", accuracy_score(y_te, y_pred_sm))
print(confusion_matrix(y_te, y_pred_sm))
print("Macro F1:", f1_score(y_te, y_pred_sm, average='macro'))
print("Micro F1:", f1_score(y_te, y_pred_sm, average='micro'))

# Визуализация вероятностей для одного примера
example_idx = 0
probs = softmax.predict_proba(X_te[example_idx:example_idx+1])[0]
plt.bar(digits.target_names, probs)
plt.xlabel('Класс')
plt.ylabel('Вероятность')
plt.title(f'Истинный класс: {y_te[example_idx]}')
plt.show()


# 17. Мультлебеловая классификация (multi‑label)

В задачах multi‑label классификации каждый объект может принадлежать одновременно нескольким классам, т.е. вектор меток $\mathbf{y} \in \{0,1\}^K$, и сумма его компонент не равна 1 (например, жанры фильма, теги новости). Модель строится как $K$ независимых бинарных классификаторов: для каждого класса $k$ предсказывается вероятность $p_k = \sigma(\mathbf{w}_k^T \boldsymbol{\phi}(\mathbf{x}))$, а функция потерь — сумма бинарных кросс-энтропий по всем классам:
$$
E = -\sum_{i=1}^{n} \sum_{k=1}^{K} \bigl[ y_{ik} \log p_{ik} + (1-y_{ik}) \log(1-p_{ik}) \bigr].
$$
Градиент по весам класса $k$ идентичен бинарному случаю: $\nabla_{\mathbf{w}_k} E = \Phi^T(\mathbf{p}_k - \mathbf{y}_k)$. На практике это реализуется с помощью `OneVsRestClassifier(LogisticRegression())`.

Специфические метрики: **Hamming Loss** (доля неверно предсказанных меток), **Subset Accuracy** (доля полностью правильно предсказанных наборов), macro‑ и micro‑F1.

Сгенерируем синтетические данные с несколькими метками.




In [ ]:
from sklearn.datasets import make_multilabel_classification
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import hamming_loss, accuracy_score

X_ml, y_ml = make_multilabel_classification(n_samples=1000, n_features=20, n_classes=5, n_labels=2,
                                            random_state=0)
X_tr_ml, X_te_ml, y_tr_ml, y_te_ml = train_test_split(X_ml, y_ml, test_size=0.3, random_state=0)

clf = OneVsRestClassifier(LogisticRegression(max_iter=1000))
clf.fit(X_tr_ml, y_tr_ml)
y_pred_ml = clf.predict(X_te_ml)

print("Hamming loss:", hamming_loss(y_te_ml, y_pred_ml))
print("Subset accuracy:", accuracy_score(y_te_ml, y_pred_ml))
print("Micro F1:", f1_score(y_te_ml, y_pred_ml, average='micro'))
print("Macro F1:", f1_score(y_te_ml, y_pred_ml, average='macro'))


**Итоговая таблица: Бинарная vs Многоклассовая vs Мультлебеловая**

| Характеристика | Бинарная | Многоклассовая (Softmax) | Мультлебеловая |
|----------------|----------|---------------------------|----------------|
| Число классов | 2 | $K$ взаимоисключающих | $K$ независимых |
| Выход модели | Одна вероятность $p$ | $K$ вероятностей, сумма = 1 | $K$ вероятностей, сумма произвольная |
| Функция потерь | Бинарная кросс-энтропия | Категориальная кросс-энтропия | Сумма $K$ бинарных кросс-энтропий |
| Пример задачи | Спам/не спам | Распознавание цифр | Жанры фильма |

# 18. Краткое сравнение с другими линейными классификаторами

Логистическая регрессия занимает центральное место среди линейных классификаторов. **Линейный дискриминантный анализ (LDA)** предполагает нормальность признаков внутри каждого класса и равенство ковариационных матриц. **Метод опорных векторов (SVM)** с линейным ядром максимизирует зазор и фокусируется на опорных объектах, используя hinge loss. Логистическая регрессия выигрывает в интерпретируемости (выдаёт вероятности) и устойчивости к шумам, тогда как SVM лучше при чёткой разделимости классов. На практике логистическую регрессию часто используют как baseline, а при необходимости повышения качества переходят к более сложным нелинейным моделям.

# 19. Мост к градиентному бустингу

Кросс-энтропия, лежащая в основе логистической регрессии, является также стандартной функцией потерь для бинарной классификации в градиентном бустинге (XGBoost, LightGBM, CatBoost). Различие в том, что в логистической регрессии мы строим **одну** линейную комбинацию $\mathbf{w}^T\boldsymbol{\phi}(\mathbf{x})$, оптимизируя её параметры градиентным спуском. В градиентном бустинге строится ансамбль слабых моделей (обычно деревьев), которые последовательно добавляются друг к другу. Каждое новое дерево обучается на **антиградиенте** кросс-энтропии текущей композиции — это функциональный градиентный спуск в пространстве предсказаний. Таким образом, бустинг можно рассматривать как непараметрическое обобщение логистической регрессии, где признаки и их взаимодействия автоматически конструируются деревьями.




In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb.fit(X_train_scaled, y_train)
y_pred_gb = gb.predict(X_test_scaled)
print("Gradient Boosting accuracy:", accuracy_score(y_test, y_pred_gb))


Обычно градиентный бустинг даёт более высокое качество, но логистическая регрессия остаётся прозрачной и вычислительно лёгкой моделью, идеально подходящей для начального анализа.

# 20. Заключение и резюме главы

Мы проследили путь от бинарной логистической регрессии до многоклассовых и multi‑label расширений, изучили регуляризацию, диагностику и место модели в более широком контексте машинного обучения. Логистическая регрессия — это не просто «линейная регрессия для классификации», а полноценная вероятностная модель, глубокая и гибкая.

**Итоговая таблица: аналогия с линейной регрессией**

| Компонент | Линейная регрессия | Логистическая регрессия |
|-----------|-------------------|-------------------------|
| Линейный предиктор | $\mathbf{w}^T\boldsymbol{\phi}$ | $\mathbf{w}^T\boldsymbol{\phi}$ |
| Функция связи | Тождественная ($\mu=\eta$) | Logit ($\eta = \log\frac{\mu}{1-\mu}$) |
| Распределение отклика | Нормальное | Бернулли (биномиальное) |
| Функция потерь | MSE (квадратичная) | Кросс-энтропия |
| Градиент | $\Phi^T(\Phi\mathbf{w} - \mathbf{y})$ | $\Phi^T(\boldsymbol{\sigma} - \mathbf{y})$ |
| Регуляризация | Ridge/Lasso/ElasticNet | Ridge/Lasso/ElasticNet |
| Аналитическое решение | Да | Нет (IRLS, численные методы) |

---

## Вопросы для самопроверки

### Для начинающих
1. В чём разница между $L_1$- и $L_2$-регуляризацией в логистической регрессии? Какой эффект даёт $L_1$-регуляризация при наличии большого числа признаков?
2. Что такое softmax-функция и для чего она используется?
3. Чем отличаются задачи многоклассовой (multi‑class) и мультлебеловой (multi‑label) классификации?
4. Как интерпретировать коэффициент $w_j$ в логистической регрессии? Что означает его знак и модуль?
5. Какую функцию потерь мы минимизируем в бинарной логистической регрессии? Почему она называется кросс-энтропией?
6. В чём преимущество использования `class_weight='balanced'` при несбалансированных классах?

### Для профессионалов
1. Выведите градиент категориальной кросс-энтропии и покажите, что он имеет вид $\Phi^T(\hat{\mathbf{P}} - \mathbf{Y})$.
2. Докажите, что функция кросс-энтропии строго выпукла, используя гессиан $\Phi^T D \Phi$ и свойства матрицы $D$.
3. Что такое оператор мягкого порога (soft‑thresholding) и как он применяется при $L_1$-регуляризации логистической регрессии?
4. Объясните, почему логистическая регрессия является частным случаем обобщённых линейных моделей (GLM). Перечислите её три компонента.
5. Как связана каноническая функция связи с совпадением наблюдаемой и ожидаемой информации Фишера?

## Задания повышенной сложности

1. **Градиент softmax.** Выведите в явном виде производную $\partial \hat{p}_{ik} / \partial z_{ij}$ и постройте на её основе полный градиент функции категориальной кросс-энтропии.
2. **Выпуклость.** Покажите, что матрица вторых производных (гессиан) бинарной кросс-энтропии имеет вид $\Phi^T D \Phi$ с $D = \operatorname{diag}(p_i(1-p_i))$. Докажите, что она положительно полуопределена, и при полном ранге $\Phi$ — строго положительно определена.
3. **Реализация IRLS.** Реализуйте итеративный взвешенный метод наименьших квадратов для бинарной логистической регрессии с нуля и сравните его сходимость с SGD.
4. **Асимптотика MLE.** Выведите асимптотическое распределение $\hat{\mathbf{w}}_{\text{MLE}}$ для логистической регрессии, используя разложение градиента и центральную предельную теорему.
5. **Калибровка.** Постройте калибровочную кривую для модели с $L_1$-регуляризацией, вычислите Brier score и ECE (Expected Calibration Error), затем примените изотоническую калибровку и сравните результаты.